In [8]:
library(tidyverse)
library(pheatmap)
library(data.table)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

twist_barcodes <- read_csv("/mnt/dawnccle2/melange/data/guide_library/20230130_twist_library_v3.csv") %>%
  mutate(barcodeRevcomp = sapply(barcode, reverse_complement))
all_events <- read_csv("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/MUT_all_samples_PSI_count_table_major_events.csv")

######## Look at rmats stats ########
combined_psi <- read_tsv("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/MUT_PSI_combined_output_indiv.tsv") %>%
  filter(ExonID %in% all_events$index_offset)
calculate_ratio <- function(I, S) {
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  ratio <- I_values / (I_values + S_values)
  return(paste(round(ratio,3), collapse = ","))
}

calculate_average <- function(PSI){
  PSI_values <- as.numeric(unlist(strsplit(PSI, ",")))
  average <- mean(PSI_values)
  return(round(average, 3))
}

calculate_average_count_sum <- function(I, S){
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  total_sum <- I_values + S_values
  average_count_sum <- mean(total_sum)
  return(round(average_count_sum, 0))
}

# Apply the function to the data frame
combined_psi <- combined_psi %>%
  mutate(
    PSI1 = mapply(calculate_ratio, I1, S1),
    PSI2 = mapply(calculate_ratio, I2, S2)
  ) %>% 
  mutate(
    PSI1_average = mapply(calculate_average, PSI1),
    PSI2_average = mapply(calculate_average, PSI2)
  ) %>%
  mutate(PSI_diff = PSI2_average - PSI1_average) %>% 
  mutate(
    count_sum_average1 = mapply(calculate_average_count_sum, I1, S1),
    count_sum_average2 = mapply(calculate_average_count_sum, I2, S2)
  ) %>% mutate(PSI_ratio = PSI2_average / PSI1_average) %>% 
  mutate(PSI_reverse_ratio = (1-PSI2_average)/(1-PSI1_average))

wanted_pairs <- c(
#'MUT2_CH3-1_A1_CH3-1_A2',
'MUT_CH3-1_A1_FUBP1_B12',
'MUT_CH3-1_A1_FUBP1_C5',
'MUT_CH3-1_A1_RBM10_C8',
'MUT_CH3-1_A1_RBM10_G4',
'MUT_CH3-1_A1_RBM5_A2',
'MUT_CH3-1_A1_RBM5_A3',
'MUT_CH3-1_A1_ZRSR2_F8',
'MUT_CH3-1_A1_ZRSR2_G9',
'MUT_CH3-1_A2_FUBP1_B12',
'MUT_CH3-1_A2_FUBP1_C5',
'MUT_CH3-1_A2_RBM10_C8',
'MUT_CH3-1_A2_RBM10_G4',
'MUT_CH3-1_A2_RBM5_A2',
'MUT_CH3-1_A2_RBM5_A3',
'MUT_CH3-1_A2_ZRSR2_F8',
'MUT_CH3-1_A2_ZRSR2_G9',
'MUT_K562WT_K562K700E',
'MUT_MUT-plx317_U2AF1_WT_MUT-plx317_U2AF1_Q157A',
'MUT_MUT-plx317_U2AF1_WT_MUT-plx-LacZ',
'MUT_MUT-sgCh3-1_MUT-sgRBM10',
'MUT_MUT-sgCh3-1_MUT-sgRBM5',
'MUT_MUT-sgCh3-1_MUT-sgRUBP1',
'MUT_splicelib_sgCh3_splicelib_ZRSR2',
'MUT_splicelib_U2AF1_WT_splicelib_U2AF1_S34F')

same_pairs <- c(
'MUT_CH3-1_A1_CH3-1_A2',
'MUT_FUBP1_B12_FUBP1_C5',
'MUT_RBM10_C8_RBM10_G4',
'MUT_RBM5_A2_RBM5_A3',
'MUT_ZRSR2_F8_ZRSR2_G9')



Rows: 46372 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): ID, barcode, upstreamIntronSeq, skippedExonSeq, downstreamIntronSeq...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2366344 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (6): index, mode, offset, sample, condition, index_offset
dbl (2): included_count, skipped_count

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 12763 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (6): ExonID, I1, S1, I2, S2, Folder
dbl (4): I_len, S_len, PValue, FDR

ℹ Use `spec()` to retrieve the full column specification for this data.


In [9]:

# Get sequences in folders MUT2_CH3-1_A1
all_ch3_a1 <- combined_psi %>% filter(grepl("MUT_CH3-1_A1", Folder)) 
chr3_a1_seq_count <- all_ch3_a1 %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
arrange(desc(count))

seq_to_exclude_a1 <- chr3_a1_seq_count %>% 
filter(count >=4) %>% 
pull(ExonID)

# Also exclude sequences in MUT2_CH3-1_A2
all_ch3_a2 <- combined_psi %>% filter(grepl("MUT_CH3-1_A2", Folder)) 
chr3_a2_seq_count <- all_ch3_a2 %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
arrange(desc(count))

seq_to_exclude_a2 <- chr3_a2_seq_count %>% 
filter(count >=4) %>% 
pull(ExonID)

# Also look at diff between A1 and A2
control_de <- combined_psi %>% 
filter(Folder == "MUT2_CH3-1_A1_CH3-1_A2") %>% 
filter(count_sum_average1 >30) %>% 
filter(count_sum_average2 > 30) %>% 
mutate(log2_PSI_ratio = log2(PSI_ratio), log2_PSI_reverse_ratio = log2(PSI_reverse_ratio)) %>% 
  separate(ExonID, sep = "__", into =c("index", "offset"), remove = FALSE) %>%
  separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = FALSE) %>%
  filter(abs(as.integer(skipped_exon_start)) != 1 & abs(as.integer(skipped_exon_end)) != 1) 

seq_to_exclude_control <- control_de %>% 
pull(ExonID) %>% 
unique()

all_seq_to_exclude <- c(seq_to_exclude_a1, seq_to_exclude_a2, seq_to_exclude_control)

combined_psi_filtered <- combined_psi %>% 
  filter(!ExonID %in% all_seq_to_exclude) %>% 
  # filter(Folder %in% wanted_pairs) %>%
  filter(count_sum_average1 >30) %>% 
  filter(count_sum_average2 > 30) %>% 
  mutate(log2_PSI_ratio = log2(PSI_ratio), log2_PSI_reverse_ratio = log2(PSI_reverse_ratio)) %>% 
  separate(ExonID, sep = "__", into =c("index", "offset"), remove = FALSE) %>%
  separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = FALSE) %>%
  filter(abs(as.integer(skipped_exon_start)) != 1 & abs(as.integer(skipped_exon_end)) != 1) 

######## Process and Save the Individual Sequences as Fasta ######
# Define a function to process and save sequences
process_and_save <- function(df, name) {
  # Define file paths
  upstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", name, "_upstreamIntronSeq_adj.fasta")
  skipped_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", name, "_skippedExonSeq_adj.fasta")
  downstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", name, "_downstreamIntronSeq_adj.fasta")
  
  # Write upstream intron sequences
  fileConn <- file(upstream_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["upstreamIntronSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Write skipped exon sequences
  fileConn <- file(skipped_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["skippedExonSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Write downstream intron sequences
  fileConn <- file(downstream_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["downstreamIntronSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Save CSV file
  csv_path <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT_", name, "_seq.csv")
  write_csv(df, csv_path)
  
  # Message to indicate completion
  cat("FASTA and CSV files saved for:", name, "\n")
}

# Process and save each dataset
unique_folders <- unique(combined_psi_filtered$Folder)

# Create a list of datasets for all unique folders
datasets <- list()
for (folder in unique_folders) {
   # folder_name <- gsub("MUT2_", "", folder)  # Remove the MUT2_ prefix if present
  datasets[[folder]] <- combined_psi_filtered %>% filter(Folder == folder)
}

# Print the number of datasets to be processed
cat("Processing", length(datasets), "unique datasets\n")

for (name in names(datasets)) {
  df <- datasets[[name]] %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )
  
  process_and_save(df, name)
}

cat("Processing complete for all datasets.\n")

# Make the "background sequence" file 
upstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", "background_upstreamIntronSeq_adj.fasta")
skipped_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", "background_skippedExonSeq_adj.fasta")
downstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", "background_downstreamIntronSeq_adj.fasta")

# Write upstream intron sequences
fileConn <- file(upstream_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["upstreamIntronSeq"], "\n"), file = fileConn)
})
close(fileConn)

# Write skipped exon sequences
fileConn <- file(skipped_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["skippedExonSeq"], "\n"), file = fileConn)
})
close(fileConn)

# Write downstream intron sequences
fileConn <- file(downstream_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["downstreamIntronSeq"], "\n"), file = fileConn)
})
close(fileConn)

Processing 28 unique datasets
FASTA and CSV files saved for: MUT_CH3-1_A1_CH3-1_A2 
FASTA and CSV files saved for: MUT_CH3-1_A1_FUBP1_B12 
FASTA and CSV files saved for: MUT_CH3-1_A1_FUBP1_C5 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM10_C8 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM10_G4 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM5_A2 
FASTA and CSV files saved for: MUT_CH3-1_A1_RBM5_A3 
FASTA and CSV files saved for: MUT_CH3-1_A1_ZRSR2_F8 
FASTA and CSV files saved for: MUT_CH3-1_A1_ZRSR2_G9 
FASTA and CSV files saved for: MUT_CH3-1_A2_FUBP1_B12 
FASTA and CSV files saved for: MUT_CH3-1_A2_FUBP1_C5 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM10_C8 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM10_G4 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM5_A2 
FASTA and CSV files saved for: MUT_CH3-1_A2_RBM5_A3 
FASTA and CSV files saved for: MUT_CH3-1_A2_ZRSR2_F8 
FASTA and CSV files saved for: MUT_CH3-1_A2_ZRSR2_G9 
FASTA and CSV files saved for: MUT_FUBP1_B12_FUBP1_C5 

NULL

NULL

NULL

In [10]:
# Plot the volcano plot
# save the plot
p1 <- ggplot(combined_psi_filtered, aes(log2(PSI_ratio), -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 4) +
  theme_classic() +
  ggtitle("PSI ratio")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT_volcano_plot_PSI_ratio.png", p1, width = 20, height = 20)

# Plot PSI_reverse_ratio
p2 <- ggplot(combined_psi_filtered, aes(log2(PSI_reverse_ratio), -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 4) +
  theme_classic() +
  ggtitle("PSI reverse ratio")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT_volcano_plot_PSI_reverse_ratio.png", p2, width = 20, height = 20)

# Also plot the PSI_dff
p3 <- ggplot(combined_psi_filtered, aes(PSI_diff, -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 4) +
  theme_classic() +
  ggtitle("PSI diff")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT_volcano_plot_PSI_diff.png", p3, width = 20, height = 20)
 


In [11]:
# blacklist
blacklist_seqs <- c("MUT_CH3-1_A1_CH3-1_A2",
"MUT_FUBP1_B12_FUBP1_C5",
"MUT_RBM10_C8_RBM10_G4",
"MUT_RBM5_A2_RBM5_A3",
"MUT_ZRSR2_F8_ZRSR2_G9")

# Across all samples, get log2(PSI_ratio) > 2
high_ratio <- combined_psi_filtered %>% 
filter(!Folder %in% blacklist_seqs) %>% 
filter(log2_PSI_ratio >= 3 | log2_PSI_reverse_ratio >= 3) %>% 
filter(abs(PSI_diff) >= 0.3) %>% 
arrange(desc(pmax(log2_PSI_ratio, log2_PSI_reverse_ratio))) %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )

write_csv(high_ratio, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT_high_ratio_PSI.csv")


# Save a separate file for not multiple of 3 for skippedExon_len_adj
high_ratio_not_multiple_of_3 <- high_ratio %>%
  filter(skippedExon_len_adj %% 3 != 0)

write_csv(high_ratio_not_multiple_of_3, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT_high_ratio_PSI_not_multiple_of_3.csv")


nrow(high_ratio)
# Get how many in each folder
high_ratio %>% 
group_by(Folder) %>% 
summarise(count = n()) %>% 
arrange(desc(count)) %>% 
head(20)


high_ratio %>% 
    group_by(ExonID) %>% 
    summarise(n = n(),
    samples = paste(Folder, collapse = ","),
    log_fold_change = paste(round(log2_PSI_reverse_ratio,3), collapse = ",")) %>% 
    arrange(desc(n)) %>% 
    head(20)
# Get how many unique sequences
length(unique(high_ratio$ExonID))

[1] 126

Folder,count
<chr>,<int>
MUT_K562WT_K562K700E,36
MUT_CH3-1_A1_ZRSR2_F8,16
MUT_CH3-1_A1_ZRSR2_G9,13
MUT_CH3-1_A1_FUBP1_B12,12
MUT_CH3-1_A1_RBM5_A2,9
MUT_CH3-1_A1_FUBP1_C5,7
MUT_CH3-1_A2_RBM10_G4,5
MUT_CH3-1_A1_RBM5_A3,4
MUT_CH3-1_A2_FUBP1_B12,4


ExonID,n,samples,log_fold_change
<chr>,<int>,<chr>,<chr>
ENSG00000164615.6;CAMLG;chr5-134743986-134744052-134738496-134741523-134750758-134751377__0:0:0,3,"MUT_CH3-1_A1_RBM5_A2,MUT_CH3-1_A1_ZRSR2_F8,MUT_CH3-1_A1_ZRSR2_G9","Inf,Inf,Inf"
ENSG00000075914.13;EXOSC7;chr3-44989549-44989644-44976243-44976334-44997086-44997252__0:0:0,2,"MUT_CH3-1_A2_FUBP1_B12,MUT_CH3-1_A1_FUBP1_B12","3.958,3.61"
ENSG00000082996.20;RNF13;chr3-149921133-149921227-149911977-149912083-149960055-149960136__0:0:0,2,"MUT_CH3-1_A1_RBM5_A3,MUT_CH3-1_A2_RBM5_A3","Inf,Inf"
ENSG00000105696.9;TMEM59L;chr19-18616999-18617102-18615974-18616127-18618154-18618222__0:0:0,2,"MUT_CH3-1_A1_RBM10_G4,MUT_CH3-1_A2_RBM10_G4","-0.65,-0.65"
ENSG00000111863.13;ADTRP;chr6-11770030-11770084-11768248-11768383-11778606-11778803__0:0:0,2,"MUT_CH3-1_A1_RBM5_A3,MUT_CH3-1_A1_ZRSR2_F8","6.511,6.049"
ENSG00000129116.19;PALLD;chr4-168896548-168896599-168894440-168894677-168898492-168898714__0:0:0,2,"MUT_CH3-1_A2_RBM10_G4,MUT_CH3-1_A2_RBM10_C8","-0.567,-0.558"
ENSG00000143194.13;MAEL;chr1-166991377-166991477-166989736-166989829-166992685-166992841__0:0:0,2,"MUT_CH3-1_A1_FUBP1_B12,MUT_CH3-1_A1_RBM5_A2","3.628,3.483"
ENSG00000144036.16;EXOC6B;chr2-72514633-72514680-72513131-72513252-72515042-72515126__0:0:0,2,"MUT_CH3-1_A1_ZRSR2_F8,MUT_CH3-1_A1_ZRSR2_G9","3.931,3.796"
ENSG00000183077.15;AFMID;chr17-78190969-78191060-78187316-78187433-78197134-78197560__0:0:0,2,"MUT_CH3-1_A1_FUBP1_C5,MUT_CH3-1_A2_FUBP1_C5","Inf,Inf"


[1] 111